In [1]:
!pip install imbalanced-learn seaborn -q

In [2]:
from google.colab import files

uploaded = files.upload()


Saving loan_dataset.csv.docx to loan_dataset.csv.docx


In [3]:
# =========================================================
# ALGORITHMIC CREDIT RISK MODELING PROJECT
# =========================================================

# =========================================================
# IMPORT LIBRARIES
# =========================================================

import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import SGDClassifier
from sklearn.metrics import (
    roc_auc_score,
    roc_curve,
    classification_report,
    confusion_matrix
)

from imblearn.over_sampling import SMOTE

# =========================================================
# LOAD DATASET IN CHUNKS
# =========================================================

print("Loading dataset in chunks...")

INPUT_FILE = list(uploaded.keys())[0]

CHUNK_SIZE = 100000

chunks = []

for chunk in pd.read_csv(INPUT_FILE, chunksize=CHUNK_SIZE):

    # Clean column names
    chunk.columns = chunk.columns.str.strip().str.lower()

    # Remove duplicates
    chunk.drop_duplicates(inplace=True)

    chunks.append(chunk)

# Combine chunks
df = pd.concat(chunks, ignore_index=True)

print("\nDataset Loaded Successfully")
print("Dataset Shape:", df.shape)

# =========================================================
# DATASET OVERVIEW
# =========================================================

print("\n========== DATASET INFO ==========")
print(df.info())

print("\n========== STATISTICAL SUMMARY ==========")
print(df.describe())

print("\n========== TARGET DISTRIBUTION ==========")
print(df["default"].value_counts())

# =========================================================
# HANDLE MISSING VALUES
# =========================================================

numeric_cols = df.select_dtypes(include=np.number).columns

for col in numeric_cols:
    df[col] = df[col].fillna(df[col].median())

# =========================================================
# FEATURE / TARGET SPLIT
# =========================================================

X = df.drop("default", axis=1)
y = df["default"]

# Convert categorical columns
X = pd.get_dummies(X, drop_first=True)

# =========================================================
# TRAIN TEST SPLIT
# =========================================================

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# =========================================================
# FEATURE SCALING
# =========================================================

scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# =========================================================
# HANDLE IMBALANCED DATA USING SMOTE
# =========================================================

print("\nApplying SMOTE...")

smote = SMOTE(random_state=42)

X_train_smote, y_train_smote = smote.fit_resample(
    X_train,
    y_train
)

print("\nBefore SMOTE:")
print(y_train.value_counts())

print("\nAfter SMOTE:")
print(pd.Series(y_train_smote).value_counts())

# =========================================================
# TRAIN ELASTICNET MODEL
# =========================================================

print("\nTraining ElasticNet Model...")

model = SGDClassifier(
    loss="log_loss",
    penalty="elasticnet",
    alpha=0.0001,
    l1_ratio=0.5,
    random_state=42
)

model.fit(X_train_smote, y_train_smote)

# =========================================================
# PREDICTIONS
# =========================================================

y_probs = model.predict_proba(X_test)[:, 1]

# =========================================================
# ROC-AUC SCORE
# =========================================================

roc_auc = roc_auc_score(y_test, y_probs)

print(f"\nROC-AUC Score: {roc_auc:.4f}")

# =========================================================
# ROC CURVE
# =========================================================

fpr, tpr, thresholds = roc_curve(y_test, y_probs)

plt.figure(figsize=(10,6))

plt.plot(fpr, tpr, label=f"ROC Curve (AUC = {roc_auc:.4f})")
plt.plot([0,1], [0,1], linestyle="--")

plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC-AUC Curve")

plt.legend()

plt.show()

# =========================================================
# FIND BEST THRESHOLD
# =========================================================

print("\nFinding Optimal Threshold...")

profits = []

for threshold in thresholds:

    y_pred = (y_probs >= threshold).astype(int)

    tn, fp, fn, tp = confusion_matrix(y_test, y_pred).ravel()

    # Example bank profit formula
    profit = (tp * 1000) - (fp * 500) - (fn * 700)

    profits.append(profit)

best_index = np.argmax(profits)

best_threshold = thresholds[best_index]

print(f"\nBest Threshold: {best_threshold:.4f}")
print(f"Maximum Profit Score: {profits[best_index]}")

# =========================================================
# FINAL PREDICTIONS USING BEST THRESHOLD
# =========================================================

final_predictions = (
    y_probs >= best_threshold
).astype(int)

# =========================================================
# CLASSIFICATION REPORT
# =========================================================

print("\n========== CLASSIFICATION REPORT ==========")

print(
    classification_report(
        y_test,
        final_predictions
    )
)

# =========================================================
# CONFUSION MATRIX
# =========================================================

cm = confusion_matrix(
    y_test,
    final_predictions
)

plt.figure(figsize=(6,5))

sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues"
)

plt.title("Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("Actual")

plt.show()

print("\n========== PROJECT COMPLETED ==========")

Loading dataset in chunks...


UnicodeDecodeError: 'utf-8' codec can't decode byte 0xd2 in position 16: invalid continuation byte

### Inspecting the uploaded file

Let's write the content of the `uploaded` dictionary to a file and then inspect it using shell commands to confirm its type.

In [4]:
import os

# Get the filename from the uploaded dictionary
input_filename = list(uploaded.keys())[0]

# Write the content to a file in the Colab environment
with open(input_filename, 'wb') as f:
    f.write(uploaded[input_filename])

print(f"File '{input_filename}' written to disk.")

# Check the file type using the 'file' command
print("\nFile type:")
!file {input_filename}

# Display the first few lines of the file (will be binary for .docx)
print("\nFirst 10 lines of the file:")
!head {input_filename}

File 'loan_dataset.csv.docx' written to disk.

File type:
loan_dataset.csv.docx: Microsoft Word 2007+

First 10 lines of the file:
PK     ! ߤ�lZ      [Content_Types].xml �(�                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                 ���n�0E�����Ub袪*�>�-R�{V��Ǽ��QU�
l"%3��3Vƃ�ښl	�w%�=���^i7+���-d&�0�A�6�l4��L60#�Ò�S
��	��^�[��x ����1x�p����f��#I)ʃ�Y���������*D��i")��c$���qU���~3��1��jH[{�=E����~
f?��3-���޲]�Tꓸ2�j)�,l0/%��b�
��'
��92� �����C��@�	�f1bD����q
H�6 ��ޞd!=d
"     word/theme/theme1.x

As you can see from the `file` command output, the file is identified as a "Microsoft Word 2007+" document, which is a ZIP archive format. This is why `pd.read_csv` was unable to decode it as a plain text CSV.

Please upload the actual `.csv` file for your dataset.